In [19]:
from langchain_ollama import ChatOllama
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

import pandas as pd
import re
import os

#gemma4:31b-cloud

In [ ]:
def parse_llm_response(response_text: str) -> dict:
    """
    Parses a markdown table from the LLM response into a dictionary.
    Format: { 'element_name': ('label', 'arg1', 'arg2') }
    """
    # Extract the table using regex
    table_match = re.search(r'(\| label \| element \| arg1 \| arg2 \|.*)', response_text, re.DOTALL)
    if not table_match:
        return {}

    table_str = table_match.group(1)
    lines = table_str.strip().split('\n')

    # Filter out the header and the separator line (---|---)
    data_lines = [line for line in lines if '---|---' not in line and 'label | element' not in line]

    parsed_data = {}
    for line in data_lines:
        # Split by pipe, strip whitespace, and remove empty strings from ends
        cols = [c.strip() for c in line.split('|') if c.strip()]
        if len(cols) >= 2:
            label = cols[0]
            element = cols[1]
            arg1 = cols[2] if len(cols) > 2 else ""
            arg2 = cols[3] if len(cols) > 3 else ""
            parsed_data[element] = (label, arg1, arg2)

    return parsed_data

def score_response(response: str, ground_truth: pd.DataFrame) -> tuple[float, float, float]:
    """
    Scores the model response against a ground truth DataFrame.
    Returns: (precision, recall, f1)
    """
    # Convert ground truth DataFrame to dict for fast lookup: {element: (label, arg1, arg2)}
    truth_dict = {row['element']: (row['label'], row['arg1'], row['arg2']) for _, row in ground_truth.iterrows()}

    predictions = parse_llm_response(response)

    # True Positives: Element exists and all associated columns match exactly
    tp = sum(1 for element, values in predictions.items() if element in truth_dict and values == truth_dict[element])

    # Total predictions minus correct ones
    fp = len(predictions) - tp

    # Total expected minus correct ones
    fn = len(truth_dict) - tp

    c_matrix = {"TP": tp, "FP": fp, "FN": fn}

    precision = calc_precision(c_matrix) if (tp + fp) > 0 else 0.0
    recall = calc_recall(c_matrix) if (tp + fn) > 0 else 0.0
    f1 = calculate_f1(precision, recall) if (precision + recall) > 0 else 0.0

    return precision, recall, f1

def build_eval_prompt(state: dict) -> str:
    """
    Constructs a prompt for the optimizer to improve the current prompt
    by providing results and ground truths for error analysis.
    """
    # These variables should be defined in your notebook's global scope (e.g., via pd.read_csv)
    ground_truths = {
        'library': lib_truth,
        'rental': rent_truth,
        'ntss': ntss_truth
    }

    return (
        f"Current Prompt: {state['current_prompt']}\n\n"
        f"Evaluation Results: {state['lib_eval']} {state['rental_eval']} {state['ntss_eval']}\n\n"
        f"Ground Truths: {ground_truths}\n\n"
        f"Based on the results and the ground truths, please analyze where the predictions "
        f"went wrong and provide an improved version of the prompt to increase the F1 score."
    )

def generate_confusion_matrix(pred, targets):
    """Takes model predictions and ground truths and returns a confusion matrix as a dict"""
    return { "TP":0, "TN":0, "FP":0, "FN":0 }

def calc_precision(c_matrix):
    """Given confusion matrix, calculate how precise the models predictions are"""
    return c_matrix["TP"] / (c_matrix["TP"] + c_matrix["FP"])

def calc_recall(c_matrix):
    """Given confusion matrix, calculate the models ability to identify all positive cases in a dataset"""
    return c_matrix["TP"] / (c_matrix["TP"] + c_matrix["FN"])

def calculate_f1(precision, recall):
    """Calculates F1 score for classification task"""
    return 2 * ((precision * recall) / (precision + recall)) if (precision + recall) > 0 else 0.0

def should_continue(state: dict):
    """Determines if the optimizer should run again or stop"""
    max_trials = 5
    if state["current_trial"] < max_trials:
        return "run_lib"
    return END



In [ ]:
class OptimizerState(TypedDict):
    current_prompt: str
    current_trial: int
    optimized_prompt: str
    rental_eval: dict[str,str]
    ntss_eval: dict[str,str]
    lib_eval: dict[str,str]

In [14]:
model_name = "gemma4:31b-cloud"

lib_phrases = "library, loan items, customers, member, membership card, member number, details, name, address, date of birth, subject sections, classification mark, bar code, language tapes, books, title language, level, title, author(s), current loan, bar code reader, membership, book bar code, records, number of subject sections, types of loan items, number of items, update of records, issues, shows, kept, denoted, borrow, reserved, renewed, extend, scanned, entered, read, stamped, searched, identified, French, beginner, daily, valid, two, maximum of 8, less than 8, a number of, has a title language, has a title and author(s), made up of, customer is known as a member, language tapes and books are two types of loan items"

ntss_phrases = "NTSS, service provider, business customers, create, promote, organize, run, national, international, trade shows, contact, services, involves, design, including, theme, slogan, location, duration, promoting, advertisement, organizing, creation, promotion, inviting, speakers, registering, participants, exhibitors, running, registration, setting up, booths, conference rooms, seminars, reception, distributing, trade show materials, creates, account, each, customer, record, service charges, payments received, account balances, trade show, can be regarded as, event, has, organizer, person, organization, contact information, website, belong to, one or more, domains, added, removed, attended by, types of, organization staff, invited and/or selected, observers, register, fee, prepares, runs, invited, give, keynote address, selected, evaluation, proposals, reviewed by, committee, reviewers, status, proposal, pending, review, accepted, rejected, exhibit, products, pay, size of the booth, large, medium, small, requested, rented, duration of the event, visit"

rental_phrases = "vehicle, manufacturer, price class, rental price, available, not available, rented out, purchase, repair, maintenance, disposal, car, passenger car, makes of car, model of car, transmission, automatic, manual, two, four, doors, sedan, hatchback, options, additional charge, depreciation of the rental cars, location, taken from, returned to, other forms of vehicle, customer, select, rental plan, daily unlimited miles plan, weekend savings plan, reserving, reservation, time of reservation, period of time, in person, by phone, voided, salesperson, process, archive, reservation form, file cabinet, sign, contract, block reservation, make, invoice, opened, cover, one or more, rentals, checked out, pay, sent to, company, rental charge, credit card, processed, credit card processing company, several"

lib_truth = pd.read_csv('ground_truths/library.csv')
rent_truth = pd.read_csv('ground_truths/car_rental.csv')
ntss_truth = pd.read_csv('ground_truths/ntss.csv')

model = ChatOllama(
    model=model_name,
    temperature=0
)

In [ ]:
## Nodes
def run_lib_classifier(state: dict):
    """Builds full lib prompt and asks model to classify domain phrases"""

    prompt_template = state["current_prompt"]
    full_prompt = str(prompt_template).replace("<DOMAIN PHRASES>", lib_phrases)
    model_response  = model.invoke(full_prompt)

    return {
        "lib_eval":{
            "response":model_response
        }
    }

def run_rental_classifier(state: dict):
    """Builds full lib prompt and asks model to classify domain phrases"""

    prompt_template = state["current_prompt"]
    full_prompt = str(prompt_template).replace("<DOMAIN PHRASES>", rental_phrases)
    model_response  = model.invoke(full_prompt)

    return {
        "rental_eval":{
            "response":model_response
        }
    }

def run_ntss_classifier(state: dict):
    """Builds full lib prompt and asks model to classify domain phrases"""

    prompt_template = state["current_prompt"]
    full_prompt = str(prompt_template).replace("<DOMAIN PHRASES>", ntss_phrases)
    model_response  = model.invoke(full_prompt)

    return {
        "ntss_eval":{
            "response":model_response
        }
    }

def score(state: dict):
    """Score evals for each dataset adding precision, recall, and f1 to state for each"""
    lib_data = state["lib_eval"]
    lib_p, lib_r, lib_f1 = score_response(lib_data["response"], lib_truth)

    rent_data = state["rental_eval"]
    rent_p, rent_r, rent_f1 = score_response(rent_data["response"], rent_truth)

    ntss_data = state["ntss_eval"]
    ntss_p, ntss_r, ntss_f1 = score_response(ntss_data["response"], ntss_truth)

    return {
        "lib_eval":{
            "precision":lib_p,
            "recall":lib_r,
            "f1":lib_f1
        },
        "rental_eval":{
            "precision":rent_p,
            "recall":rent_r,
            "f1":rent_f1
        },
        "ntss_eval":{
            "precision":ntss_p,
            "recall":ntss_r,
            "f1":ntss_f1
        }
    }

def optimize(state: dict):
    """Read eval data, prompt, and ground truths and optimize prompt"""
    eval_prompt = build_eval_prompt(state)
    improved_prompt = model.invoke(eval_prompt)

    return {
        "optimized_prompt":improved_prompt
    }

def checkpoint(state:dict):
    """Write optimized prompt to classification folder"""
    # located in: /prompts/auto_prompt_evo/classification/t{num_trial}.md
    # Need to understand what the current trial is
    with open(f"t{state["current_trial"]}", 'w', encoding='utf-8') as f:
        f.write(state["optimized_prompt"])

    return {"current_trial": state["current_trial"] + 1}

In [ ]:
# 1. Define the Graph
workflow = StateGraph(OptimizerState)

workflow.add_node("run_lib", run_lib_classifier)
workflow.add_node("run_rental", run_rental_classifier)
workflow.add_node("run_ntss", run_ntss_classifier)
workflow.add_node("score", score)
workflow.add_node("optimize", optimize)
workflow.add_node("checkpoint", checkpoint)

workflow.add_edge(START, "run_lib")
workflow.add_edge("run_lib", "run_rental")
workflow.add_edge("run_rental", "run_ntss")
workflow.add_edge("run_ntss", "score")
workflow.add_edge("score", "optimize")
workflow.add_edge("optimize", "checkpoint")
workflow.add_conditional_edges(
    "checkpoint",
    should_continue,
    {
        "run_lib":"run_lib",
        "END":END
    }
)

agent = workflow.compile()

# 2. Run the Optimizer
# Pulling initial prompt from the specified file
prompt_path = "prompts/auto_prompt_evo/classification/t1"
try:
    with open(prompt_path, 'r', encoding='utf-8') as f:
        initial_prompt = f.read()
    print("Successfully loaded prompt from:", prompt_path)
except FileNotFoundError:
    print(f"Error: Could not find file at {prompt_path}. Please check the path.")
    initial_prompt = "Fallback classification prompt... <DOMAIN PHRASES>"

inputs = {
    "current_prompt": initial_prompt,
    "current_trial": 1,
    "lib_eval": {}, "rental_eval": {}, "ntss_eval": {}
}

result = agent.invoke(inputs)
print("\n--- Optimization Result ---")
print("Optimized Prompt:\n", result['optimized_prompt'])

# Optional: Visualize Graph
display(Image(agent.get_graph().draw_mermaid_png()))
